# 베이지안 통계 03. MCMC 직관

## 학습 목표
- 핵심 정의와 정리를 자신의 말로 설명한다.
- 기본 개념 예제를 손계산으로 확인한다.
- Python 코드와 시각화를 통해 직관을 검산한다.

## 핵심 개념
- 확률과정
- 랜덤워크
- Poisson 과정
- 마팅게일

## 기본 개념 예제
목표분포를 따라 움직이는 무작위 경로를 본다.

## 실제 응용 예제
직접 계산하기 어려운 사후분포를 샘플로 근사한다.

## 이론 정리

### 정의와 관점
- 확률과정은 시간 또는 공간 인덱스마다 확률변수를 배치한 무작위 함수다.
- 랜덤워크는 독립 증분을 누적한 이산 시간 과정이다.
- Poisson 과정은 독립적인 사건 도착 시간을 모델링하는 대표적 계수과정이다.
- 마팅게일은 현재 정보까지 보았을 때 미래의 조건부기댓값이 현재값과 같은 공정 게임 모델이다.

### 핵심 명제와 정리
- Markov 성질은 미래가 과거 전체가 아니라 현재 상태에만 의존함을 뜻한다.
- Poisson 과정의 대기시간은 지수분포이며, 독립증분과 정상증분을 가진다.
- 마팅게일 정리는 무차익 금융모형과 정지시간 분석의 기반이 된다.
- Donsker 정리는 적절히 스케일한 랜덤워크가 Brownian 운동으로 수렴함을 설명한다.

### 계산과 학습 절차
- 시간 인덱스, 상태공간, 전이규칙, filtration을 명확히 정의한다.
- 샘플 경로를 여러 개 그려 평균 경향과 경로별 변동성을 구분한다.
- Markov chain은 전이행렬, 정상분포, 흡수상태를 먼저 계산한다.
- 확률과정의 모형 적합에서는 독립증분, 정상성, 분산 증가율을 점검한다.

### 자주 생기는 오해
- 한 경로의 움직임은 분포 전체의 성질을 대표하지 않을 수 있다.
- Markov 성질은 기억이 전혀 없다는 뜻이 아니라 현재 상태에 필요한 정보가 압축되어 있다는 뜻이다.
- 마팅게일은 평균적으로 공정하다는 뜻이지 개별 경로가 안정적이라는 뜻은 아니다.

### 증명으로 연결하기
- 확률과정 증명은 조건부기댓값과 filtration에 대한 가측성을 꼼꼼히 다룬다.
- 정지시간 정리는 언제 멈출 수 있는지가 정보 구조와 맞는지 확인해야 한다.
- 극한과정 증명은 유한차원분포 수렴과 tightness를 분리해 다룬다.

### 이 챕터에서 꼭 확인할 질문
- 기본 개념 예제 "목표분포를 따라 움직이는 무작위 경로를 본다."에서 실제로 사용한 정의는 무엇인가?
- 응용 예제 "직접 계산하기 어려운 사후분포를 샘플로 근사한다."에서 어떤 가정이 현실을 단순화하고 있는가?
- 코드가 연속 대상을 이산화한다면, 격자나 표본 수를 바꾸어도 결론이 유지되는가?
- 손계산 가능한 작은 사례와 노트북 결과가 같은 결론을 주는가?

## 0. 실행 준비

아래 셀은 프로젝트 루트의 `common/math_viz.py`를 찾아서 현재 챕터의 출력 폴더를 자동으로 설정합니다. Jupyter Lab을 프로젝트 루트에서 열면 가장 안정적으로 동작합니다.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display


CHAPTER_RELATIVE_DIR = Path("4학년_심화_과목과_연구_주제/10_베이지안_통계/ch03_MCMC_직관")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "common" / "math_viz.py").exists():
            return candidate
    raise RuntimeError("common/math_viz.py를 찾지 못했습니다. Jupyter Lab을 프로젝트 루트에서 열어 주세요.")


ROOT = find_project_root(Path.cwd().resolve())
NOTEBOOK_DIR = ROOT / CHAPTER_RELATIVE_DIR
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
sys.path.insert(0, str(ROOT / "common"))

from math_viz import PROFILES, run_profile

PROFILE = "stochastic_process"
TITLE = "베이지안 통계 - MCMC 직관"
CONCEPT_EXAMPLE = "목표분포를 따라 움직이는 무작위 경로를 본다."
APPLICATION_EXAMPLE = "직접 계산하기 어려운 사후분포를 샘플로 근사한다."

print("project root:", ROOT)
print("chapter dir:", NOTEBOOK_DIR)
print("profile:", PROFILE)

## 1. 이번 챕터의 시각화 코드 읽기

먼저 실제로 실행될 함수를 확인합니다. 코드를 읽으면서 입력값, 이산화 방식, 그래프가 의미하는 수학적 대상을 표시해 보세요.

In [ ]:
import inspect

print(inspect.getsource(PROFILES[PROFILE]))

## 2. 실행하고 결과 확인하기

아래 셀을 실행하면 `outputs/visualization.png`가 생성되고, 노트북 안에도 바로 표시됩니다.

In [ ]:
run_profile(
    profile=PROFILE,
    title=TITLE,
    concept=CONCEPT_EXAMPLE,
    application=APPLICATION_EXAMPLE,
    output_dir=OUTPUT_DIR,
)

display(Image(filename=str(OUTPUT_DIR / "visualization.png")))

## 3. 변형 실험

- 표본 수, 격자 크기, 초기값, 학습률, 경계조건 중 하나를 바꿔 보세요.
- 그림이 안정적으로 유지되는 범위와 결론이 바뀌는 범위를 나누어 적어 보세요.
- 손계산 가능한 작은 예제를 만들어 코드 결과와 비교해 보세요.

In [ ]:
# 여기에 자신만의 변형 실험을 작성하세요.
# 예: common/math_viz.py에서 위에 출력된 함수의 파라미터를 복사해 와서
#     표본 수, 구간, 초기값 등을 바꾼 뒤 다시 그려 볼 수 있습니다.
